# Exploratory Data Analysis - Developer Compensation

**Hypothesis:** Professional experience, geographic location, and developer role are the dominant predictors of annual developer compensation, outweighing formal education level and specific technology choices.

**Target variable:** `ConvertedCompYearly` (compensation converted to yearly USD equivalent)

**Dataset:** [Stack Overflow 2025 Developer Survey](https://survey.stackoverflow.co/)

## Section 0 - Setup and Data Loading

In [103]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Short labels for Plotly axes (survey uses long official country names).
COUNTRY_PLOT_LABELS = {
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "United States of America": "United States",
}

_EDLEVEL_RAW_TO_SHORT = {
    "Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's",
    "Associate degree (A.A., A.S., etc.)": "Associate",
    "Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)": "Bachelor's",
    "Some college/university study without earning a degree": "Some college",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary",
    "Other (please specify):": "Other",
    "Primary/elementary school": "Primary",
}


def _normalize_edlevel(value):
    if pd.isna(value):
        return value
    return _EDLEVEL_RAW_TO_SHORT.get(value, str(value))

In [76]:
csv_path = (
    Path.cwd().parents[1]
    / "data"
    / "stack-overflow-developer-survey-2025"
    / "survey_results_public_2025.csv"
)

COLUMNS = [
    "MainBranch",
    "Age",
    "EdLevel",
    "Employment",
    "WorkExp",
    "YearsCode",
    "DevType",
    "OrgSize",
    "ICorPM",
    "RemoteWork",
    "Industry",
    "Country",
    "Currency",
    "CompTotal",
    "ConvertedCompYearly",
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
]

raw = pd.read_csv(csv_path, usecols=COLUMNS, low_memory=False)
print(f"Raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

Raw dataset: 49,191 rows x 18 columns


In [77]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   MainBranch              49191 non-null  object 
 1   Age                     49191 non-null  object 
 2   EdLevel                 48149 non-null  object 
 3   Employment              48339 non-null  object 
 4   WorkExp                 42893 non-null  float64
 5   YearsCode               43042 non-null  float64
 6   DevType                 43680 non-null  object 
 7   OrgSize                 34178 non-null  object 
 8   ICorPM                  33243 non-null  object 
 9   RemoteWork              33780 non-null  object 
 10  Industry                33642 non-null  object 
 11  Country                 35437 non-null  object 
 12  Currency                35437 non-null  object 
 13  CompTotal               24866 non-null  float64
 14  LanguageHaveWorkedWith  31671 non-null

In [78]:
missing_pct = (raw.isna().mean() * 100).sort_values(ascending=False).round(1)
missing_pct.to_frame("% missing")

,% missing
ConvertedCompYearly,51.3
PlatformHaveWorkedWith,50.7
CompTotal,49.5
DatabaseHaveWorkedWith,48.1
LanguageHaveWorkedWith,35.6
ICorPM,32.4
Industry,31.6
RemoteWork,31.3
OrgSize,30.5
Country,28.0


## Section 1 - Data Cleaning and Validation

All heuristic filters are applied sequentially. A running count of rows removed at each step is printed at the end for full transparency.

In [79]:
cleaning_log = []

def log_step(name: str, before: int, after: int) -> None:
    cleaning_log.append({"step": name, "removed": before - after, "remaining": after})
    print(f"{name}: removed {before - after:,} rows -> {after:,} remaining")

df = raw.copy()
n_start = len(df)
print(f"Starting rows: {n_start:,}")

Starting rows: 49,191


### Step 1 - Respondent scope filter

In [80]:
n_before = len(df)
df = df[df["MainBranch"] == "I am a developer by profession"]
log_step("Keep professional developers only", n_before, len(df))

n_before = len(df)
df = df.dropna(subset=["ConvertedCompYearly"])
log_step("Drop missing ConvertedCompYearly", n_before, len(df))

Keep professional developers only: removed 11,724 rows -> 37,467 remaining
Drop missing ConvertedCompYearly: removed 17,272 rows -> 20,195 remaining


### Step 2 - Logical impossibility checks

In [81]:
AGE_UPPER_BOUND = {
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 80,
    "Prefer not to say": np.nan,
}

df["_age_upper"] = df["Age"].map(AGE_UPPER_BOUND)

n_before = len(df)
mask = df["_age_upper"].notna() & df["WorkExp"].notna()
impossible_work = mask & (df["WorkExp"] > df["_age_upper"])
df = df[~impossible_work]
log_step("WorkExp > Age upper bound", n_before, len(df))

n_before = len(df)
mask = df["_age_upper"].notna() & df["YearsCode"].notna()
impossible_code = mask & (df["YearsCode"] > df["_age_upper"])
df = df[~impossible_code]
log_step("YearsCode > Age upper bound", n_before, len(df))

n_before = len(df)
mask = df["WorkExp"].notna() & df["YearsCode"].notna()
suspicious_gap = mask & (df["WorkExp"] > df["YearsCode"] + 5)
df = df[~suspicious_gap]
log_step("WorkExp > YearsCode + 5 (suspicious gap)", n_before, len(df))

df = df.drop(columns=["_age_upper"])

WorkExp > Age upper bound: removed 8 rows -> 20,187 remaining
YearsCode > Age upper bound: removed 4 rows -> 20,183 remaining
WorkExp > YearsCode + 5 (suspicious gap): removed 325 rows -> 19,858 remaining


### Step 3 - Compensation sanity bounds

In [82]:
COMP_MIN = 1_000
COMP_MAX = 500_000

n_before = len(df)
df = df[df["ConvertedCompYearly"] >= COMP_MIN]
log_step(f"ConvertedCompYearly < ${COMP_MIN:,}", n_before, len(df))

n_before = len(df)
df = df[df["ConvertedCompYearly"] <= COMP_MAX]
log_step(f"ConvertedCompYearly > ${COMP_MAX:,}", n_before, len(df))

ConvertedCompYearly < $1,000: removed 456 rows -> 19,402 remaining
ConvertedCompYearly > $500,000: removed 147 rows -> 19,255 remaining


### Cleaning summary

In [83]:
summary = pd.DataFrame(cleaning_log)
summary.loc[len(summary)] = {
    "step": "TOTAL REMOVED",
    "removed": n_start - len(df),
    "remaining": len(df),
}
summary

,step,removed,remaining
0,Keep professional developers only,11724,37467
1,Drop missing ConvertedCompYearly,17272,20195
2,WorkExp > Age upper bound,8,20187
3,YearsCode > Age upper bound,4,20183
4,WorkExp > YearsCode + 5 (suspicious gap),325,19858
5,"ConvertedCompYearly < $1,000",456,19402
6,"ConvertedCompYearly > $500,000",147,19255
7,TOTAL REMOVED,29936,19255


## Section 2 - Understanding the Target Variable

### Q1: What is the distribution of `ConvertedCompYearly`?

In [84]:
comp = df["ConvertedCompYearly"]

print(f"Count:    {comp.count():,}")
print(f"Mean:     ${comp.mean():,.0f}")
print(f"Median:   ${comp.median():,.0f}")
print(f"Std:      ${comp.std():,.0f}")
print(f"Skewness: {comp.skew():.2f}")
print(f"Kurtosis: {comp.kurtosis():.2f}")

Count:    19,255
Mean:     $92,779
Median:   $79,062
Std:      $70,960
Skewness: 1.59
Kurtosis: 4.10


In [85]:
fig = px.histogram(
    df, x="ConvertedCompYearly", nbins=100,
    title="Distribution of Annual Compensation (USD)",
    labels={"ConvertedCompYearly": "Annual Compensation (USD)"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [51]:
df["LogComp"] = np.log10(df["ConvertedCompYearly"])

fig = px.histogram(
    df, x="LogComp", nbins=80,
    title="Distribution of log10(Annual Compensation)",
    labels={"LogComp": "log10(USD)"},
)
fig.update_layout(bargap=0.05)
fig.show()

### Q2: Outlier analysis - percentile breakdown

In [86]:
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
ptable = comp.quantile(percentiles).to_frame("USD")
ptable["USD"] = ptable["USD"].map("${:,.0f}".format)
ptable.index = [f"{p:.0%}" for p in percentiles]
ptable.index.name = "Percentile"
ptable

,USD
Percentile,
1%,"$1,912"
5%,"$6,887"
10%,"$15,153"
25%,"$44,543"
50%,"$79,062"
75%,"$123,993"
90%,"$181,000"
95%,"$225,000"
99%,"$350,000"


### Q3: How does salary vary by country?

In [87]:
top_countries = df["Country"].value_counts().head(20).index
df_top = df[df["Country"].isin(top_countries)].copy()
df_top["Country_plot"] = df_top["Country"].replace(COUNTRY_PLOT_LABELS)

country_order = (
    df_top.groupby("Country_plot")["ConvertedCompYearly"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_top, x="ConvertedCompYearly", y="Country_plot",
    category_orders={"Country_plot": country_order},
    title="Annual Compensation by Country (Top 20 by respondent count)",
    labels={
        "ConvertedCompYearly": "Annual Compensation (USD)",
        "Country_plot": "Country",
    },
)
fig.update_layout(height=700)
fig.show()

In [88]:
country_stats = (
    df_top.groupby("Country_plot")["ConvertedCompYearly"]
    .agg(["median", "mean", "count"])
    .sort_values("median", ascending=False)
    .rename(columns={"median": "Median USD", "mean": "Mean USD", "count": "N"})
    .rename_axis("Country")
)
country_stats["Median USD"] = country_stats["Median USD"].map("${:,.0f}".format)
country_stats["Mean USD"] = country_stats["Mean USD"].map("${:,.0f}".format)
country_stats

,Median USD,Mean USD,N
Country,,,
United States,"$150,000","$163,957",4124
Switzerland,"$142,592","$148,463",330
Denmark,"$100,777","$99,206",196
Australia,"$98,164","$108,912",465
United Kingdom,"$95,299","$108,646",1231
Canada,"$91,198","$102,613",754
Germany,"$82,254","$85,855",1738
Netherlands,"$81,210","$90,192",512
Austria,"$79,238","$81,106",226


### Q3b: Nominal vs. PPP-adjusted salary by country

`ConvertedCompYearly` converts via market exchange rates, but purchasing power differs dramatically across countries. PPP-adjusted values show what the salary actually *buys* locally.

PPP conversion factors below are from the [World Bank](https://data.worldbank.org/indicator/PA.NUS.PPP) (GDP-based, 2023 - latest available). A factor > 1 means the local currency buys less per USD-equivalent (i.e., things are cheaper domestically).

In [89]:
PPP_FACTORS = {
    "United States of America": 1.00,
    "Germany": 0.78,
    "India": 0.29,
    "United Kingdom of Great Britain and Northern Ireland": 0.80,
    "France": 0.79,
    "Canada": 0.83,
    "Ukraine": 0.22,
    "Poland": 0.48,
    "Netherlands": 0.82,
    "Italy": 0.73,
    "Brazil": 0.38,
    "Australia": 0.96,
    "Spain": 0.67,
    "Sweden": 0.89,
    "Switzerland": 1.22,
    "Czech Republic": 0.48,
    "Austria": 0.80,
    "Romania": 0.41,
    "Belgium": 0.82,
    "Denmark": 0.88,
}

df_top["PPP_Factor"] = df_top["Country"].map(PPP_FACTORS)
df_top["PPP_CompYearly"] = df_top["ConvertedCompYearly"] / df_top["PPP_Factor"]

In [90]:
nominal = df_top.groupby("Country_plot")["ConvertedCompYearly"].median()
ppp = df_top.groupby("Country_plot")["PPP_CompYearly"].median()

compare = pd.DataFrame({"Nominal USD": nominal, "PPP-adjusted USD": ppp}).dropna()
compare = compare.sort_values("Nominal USD", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=compare.index, x=compare["Nominal USD"],
    name="Nominal (market rate)", orientation="h",
))
fig.add_trace(go.Bar(
    y=compare.index, x=compare["PPP-adjusted USD"],
    name="PPP-adjusted", orientation="h",
))
fig.update_layout(
    barmode="group",
    title="Median Salary: Nominal vs. PPP-adjusted (Top 20 Countries)",
    xaxis_title="USD",
    yaxis_title="Country",
    height=700,
)
fig.show()

In [91]:
compare_display = compare.copy()
compare_display["Nominal Rank"] = compare_display["Nominal USD"].rank(ascending=False).astype(int)
compare_display["PPP Rank"] = compare_display["PPP-adjusted USD"].rank(ascending=False).astype(int)
compare_display["Rank Shift"] = compare_display["Nominal Rank"] - compare_display["PPP Rank"]
compare_display = compare_display.sort_values("PPP Rank")
compare_display["Nominal USD"] = compare_display["Nominal USD"].map("${:,.0f}".format)
compare_display["PPP-adjusted USD"] = compare_display["PPP-adjusted USD"].map("${:,.0f}".format)
compare_display

,Nominal USD,PPP-adjusted USD,Nominal Rank,PPP Rank,Rank Shift
Country_plot,,,,,
United States,"$150,000","$150,000",1,1,0
Ukraine,"$32,786","$149,027",17,2,15
Romania,"$58,007","$141,480",15,3,12
Czech Republic,"$64,592","$134,567",11,4,7
Poland,"$62,767","$130,765",14,5,9
United Kingdom,"$95,299","$119,124",5,6,-1
Switzerland,"$142,592","$116,879",2,7,-5
Denmark,"$100,777","$114,519",3,8,-5
Canada,"$91,198","$109,877",6,9,-3


## Section 3 - Testing the Hypothesis

### Q4: Is there a monotonic relationship between professional experience and salary?

In [92]:
df_exp = df.dropna(subset=["WorkExp"])

r_spearman, p_spearman = stats.spearmanr(df_exp["WorkExp"], df_exp["ConvertedCompYearly"])
print(f"Spearman r = {r_spearman:.3f}  (p = {p_spearman:.2e})")

Spearman r = 0.452  (p = 0.00e+00)


In [93]:
bins = [0, 2, 5, 10, 15, 20, 30, 50]
labels = ["0-2", "3-5", "6-10", "11-15", "16-20", "21-30", "31-50"]
df_exp["ExpBand"] = pd.cut(df_exp["WorkExp"], bins=bins, labels=labels, right=True)

fig = px.box(
    df_exp, x="ExpBand", y="ConvertedCompYearly",
    title="Annual Compensation by Experience Band",
    labels={"ExpBand": "Years of Professional Experience", "ConvertedCompYearly": "USD"},
    category_orders={"ExpBand": labels},
)
fig.show()

/var/folders/vs/_r7c0zhs07g_2l34drz17qgh0000gn/T/ipykernel_24779/3992742114.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exp["ExpBand"] = pd.cut(df_exp["WorkExp"], bins=bins, labels=labels, right=True)


### Q5: How does salary differ across developer roles?

In [94]:
role_salary = (
    df.dropna(subset=["DevType"])
    .assign(DevType=lambda d: d["DevType"].str.split(";"))
    .explode("DevType")
    .groupby("DevType")["ConvertedCompYearly"]
    .agg(["median", "count"])
    .query("count >= 30")
    .sort_values("median", ascending=True)
)

fig = px.bar(
    role_salary.reset_index(), x="median", y="DevType",
    orientation="h",
    title="Median Salary by Developer Role (min 30 respondents)",
    labels={"median": "Median Annual Compensation (USD)", "DevType": ""},
    text="count",
)
fig.update_layout(height=700)
fig.show()

### Q6: Does education level affect compensation, and does the effect hold when controlling for experience?

In [104]:
df_ed = df.dropna(subset=["EdLevel"]).copy()
df_ed["EdLevel"] = df_ed["EdLevel"].map(_normalize_edlevel)

ed_order = (
    df_ed.groupby("EdLevel")["ConvertedCompYearly"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

fig = px.box(
    df_ed,
    x="ConvertedCompYearly",
    y="EdLevel",
    category_orders={"EdLevel": ed_order},
    title="Compensation by education",
    labels={"EdLevel": "Education", "ConvertedCompYearly": "Annual comp, USD"},
)
fig.update_layout(height=520, width=900)
fig.show()

In [105]:
df_ed_exp = df_ed.dropna(subset=["WorkExp"]).copy()
df_ed_exp["ExpBand"] = pd.cut(
    df_ed_exp["WorkExp"],
    bins=[0, 5, 15, 50],
    labels=["0-5 y", "6-15 y", "16+ y"],
    right=True,
)

ed_order_exp = (
    df_ed_exp.groupby("EdLevel")["ConvertedCompYearly"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

fig = px.box(
    df_ed_exp,
    x="ConvertedCompYearly",
    y="EdLevel",
    color="ExpBand",
    category_orders={
        "ExpBand": ["0-5 y", "6-15 y", "16+ y"],
        "EdLevel": ed_order_exp,
    },
    title="Compensation by education and experience",
    labels={
        "EdLevel": "Education",
        "ConvertedCompYearly": "Annual comp, USD",
        "ExpBand": "Years exp",
    },
)
fig.update_layout(height=560, width=950)
fig.show()

### Q7: Does remote work arrangement affect salary?

In [106]:
REMOTE_WORK_SHORT = {
    "Remote": "Remote",
    "In-person": "In-person",
    "Hybrid (some in-person, leans heavy to flexibility)": "Hybrid · flex",
    "Hybrid (some remote, leans heavy to in-person)": "Hybrid · office",
    "Your choice (very flexible, you can come in when you want or just as needed)": "Your choice",
}

df_remote = df.dropna(subset=["RemoteWork"]).copy()
df_remote["RemoteShort"] = df_remote["RemoteWork"].map(REMOTE_WORK_SHORT)

remote_order = (
    df_remote.groupby("RemoteShort")["ConvertedCompYearly"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.violin(
    df_remote,
    x="RemoteShort",
    y="ConvertedCompYearly",
    box=True,
    points=False,
    category_orders={"RemoteShort": remote_order},
    title="Annual compensation by work arrangement",
    labels={"RemoteShort": "", "ConvertedCompYearly": "USD"},
)
fig.update_layout(height=500, xaxis_tickangle=0)
fig.show()

### Q8: Does company size correlate with higher compensation?

In [107]:
ORG_ORDER = [
    "Just me - I am a freelancer, sole proprietor, etc.",
    "Less than 20 employees",
    "20 to 99 employees",
    "100 to 499 employees",
    "500 to 999 employees",
    "1,000 to 4,999 employees",
    "5,000 to 9,999 employees",
    "10,000 or more employees",
]

df_org = df[df["OrgSize"].isin(ORG_ORDER)].copy()

fig = px.box(
    df_org,
    x="ConvertedCompYearly",
    y="OrgSize",
    category_orders={"OrgSize": ORG_ORDER},
    title="Annual compensation by organisation size",
    labels={"OrgSize": "", "ConvertedCompYearly": "USD"},
)
fig.update_layout(height=500, yaxis_autorange="reversed")
fig.show()

### Q9: Are `WorkExp` and `YearsCode` redundant?

In [108]:
df_both = df.dropna(subset=["WorkExp", "YearsCode"])

r_pearson, _ = stats.pearsonr(df_both["WorkExp"], df_both["YearsCode"])
r_sp, _ = stats.spearmanr(df_both["WorkExp"], df_both["YearsCode"])
print(f"Pearson  r = {r_pearson:.3f}")
print(f"Spearman p = {r_sp:.3f}")

fig = px.density_heatmap(
    df_both, x="WorkExp", y="YearsCode",
    title=f"WorkExp vs. YearsCode (Pearson r = {r_pearson:.3f})",
    labels={"WorkExp": "Professional Experience (years)", "YearsCode": "Total Coding (years)"},
)
fig.show()

Pearson  r = 0.918
Spearman p = 0.906


### Q10: Country x Education interaction on salary

In [110]:
TOP5 = df["Country"].value_counts().head(5).index.tolist()

df_interact = df[df["Country"].isin(TOP5)].copy()
df_interact["EdLevel"] = df_interact["EdLevel"].map(_normalize_edlevel)
df_interact = df_interact.dropna(subset=["EdLevel"])

heatmap_data = (
    df_interact.groupby(["Country", "EdLevel"])["ConvertedCompYearly"]
    .median()
    .unstack("EdLevel")
    .loc[TOP5]
)

y_labels = [COUNTRY_PLOT_LABELS.get(c, c) for c in heatmap_data.index]

fig = px.imshow(
    heatmap_data.values,
    x=list(heatmap_data.columns),
    y=y_labels,
    text_auto="$,.0f",
    color_continuous_scale="Viridis",
    title="Median salary: country x education (top 5 countries)",
    labels={"color": "Median USD"},
)
fig.update_layout(height=400)
fig.show()

## Section 4 - Exploratory

### Q11: What tech stacks are associated with the highest salaries?

In [111]:
df_lang = df.dropna(subset=["LanguageHaveWorkedWith"]).copy()

q1_threshold = df_lang["ConvertedCompYearly"].quantile(0.25)
q4_threshold = df_lang["ConvertedCompYearly"].quantile(0.75)

def top_langs(subset, label):
    return (
        subset.assign(lang=subset["LanguageHaveWorkedWith"].str.split(";"))
        .explode("lang")
        ["lang"]
        .value_counts(normalize=True)
        .head(15)
        .to_frame("share")
        .assign(quartile=label)
        .reset_index()
    )

bottom_q = top_langs(df_lang[df_lang["ConvertedCompYearly"] <= q1_threshold], "Bottom 25%")
top_q = top_langs(df_lang[df_lang["ConvertedCompYearly"] >= q4_threshold], "Top 25%")
lang_compare = pd.concat([bottom_q, top_q])

fig = px.bar(
    lang_compare, x="share", y="lang", color="quartile",
    barmode="group", orientation="h",
    title="Top 15 Languages: Highest vs. Lowest Salary Quartile",
    labels={"share": "Share of developers", "lang": "", "quartile": "Salary Quartile"},
)
fig.update_layout(height=550)
fig.show()

### Q12: IC vs. People Manager salary premium

In [112]:
df_ic = df.dropna(subset=["ICorPM"]).copy()

fig = px.box(
    df_ic, x="ICorPM", y="ConvertedCompYearly",
    title="Annual Compensation: Individual Contributor vs. People Manager",
    labels={"ICorPM": "", "ConvertedCompYearly": "USD"},
)
fig.show()

In [35]:
df_ic_exp = df_ic.dropna(subset=["WorkExp"]).copy()
df_ic_exp["ExpBand"] = pd.cut(
    df_ic_exp["WorkExp"],
    bins=[0, 5, 15, 50],
    labels=["0-5 yrs", "6-15 yrs", "16+ yrs"],
    right=True,
)

fig = px.box(
    df_ic_exp, x="ExpBand", y="ConvertedCompYearly", color="ICorPM",
    category_orders={"ExpBand": ["0-5 yrs", "6-15 yrs", "16+ yrs"]},
    title="IC vs. Manager Salary by Experience Band",
    labels={"ExpBand": "Experience", "ConvertedCompYearly": "USD", "ICorPM": ""},
)
fig.show()

### Q13: How does industry relate to compensation?

In [113]:
top_industries = df["Industry"].value_counts().head(10).index
df_ind = df[df["Industry"].isin(top_industries)].copy()

ind_order = (
    df_ind.groupby("Industry")["ConvertedCompYearly"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_ind, x="ConvertedCompYearly", y="Industry",
    category_orders={"Industry": ind_order},
    title="Annual Compensation by Industry (Top 10 by respondent count)",
    labels={"ConvertedCompYearly": "Annual Compensation (USD)", "Industry": ""},
)
fig.update_layout(height=500)
fig.show()

## Section 5 - Summary of Key Findings

### Target variable (Q1–Q2)
- `ConvertedCompYearly` is **right-skewed** (skewness 1.59, kurtosis 4.10); log-transformation yields a near-symmetric distribution suitable for regression.
- After cleaning, n = 19,255 professional developers with salaries in $1k–$500k.
- Median $79,062 — substantially below the mean ($92,779), confirming the long right tail.
- Middle 50 % of developers earn between $44.5k and $124k (IQR ≈ $79k).

### Q3 — How does salary vary by country?
- Geography is the **single strongest predictor**. The US leads with a median of $150k; India sits at $19.5k — a **7.7× gap**.
- The US box plot stands apart: its entire IQR (~$100k–$180k) sits above the whisker of most European countries. Switzerland ($143k) is the only country whose median is close.
- English-speaking and Nordic countries cluster at the top (US, Switzerland, Denmark, Australia, UK, Canada — all above $90k median).
- Eastern European markets (Ukraine $33k, Romania $58k, Poland $63k) and emerging economies (Brazil $33k, India $20k) form the lower tier.
- Outlier density increases with country wealth — the US and UK show the densest scatter above $300k, while lower-income countries have almost none.

### Q3b — Nominal vs PPP-adjusted salary
- PPP adjustment compresses the range dramatically. The biggest winners are **Ukraine** (nominal $33k → PPP ~$149k), **India** ($20k → ~$67k), and **Poland** ($63k → ~$131k).
- The US still leads under PPP, but Switzerland drops below it — its high cost of living erodes its nominal advantage.
- Several Eastern European countries (Czech Republic, Poland, Romania) leapfrog Western European peers (France, Italy) after PPP adjustment, revealing that their developers enjoy more purchasing power than nominal figures suggest.
- Key takeaway for the student advisory use-case: recommending countries by raw salary alone is misleading.

### Q4 — Is there a monotonic relationship between experience and salary?
- Spearman r = 0.452 (p ≈ 0) — a strong, statistically significant monotonic relationship.
- Medians rise steadily: 0–2 y ~$30k → 3–5 y ~$60k → 6–10 y ~$90k → 11–15 y ~$100k → 16–20 y ~$115k → 21–30 y ~$120k → 31–50 y ~$150k.
- The steepest salary gains occur in the **first 10 years**; after that, medians continue rising but the increments flatten — consistent with diminishing returns on experience.
- Variance (box height and outlier spread) **increases with experience**: early-career bands are tightly packed around low salaries, while senior bands have much wider IQRs, reflecting diverging career paths (management vs IC, geography, etc.).
- Every experience band has outliers reaching the $500k ceiling, but these become proportionally more common from the 6–10 y band onward.

### Q5 — How does salary differ across developer roles?
- Engineering managers ($135k, n=483) earn the highest median, followed by senior executives ($125k, n=185) and cloud infrastructure engineers ($110k, n=191).
- The bottom tier includes students ($17.4k), academic researchers ($43.3k), and project managers ($59k).
- The spread from top to bottom role is roughly **8×** (engineering manager vs student).
- Infrastructure, security, and leadership roles consistently outpay application-development roles (full-stack at ~$75k, front-end at ~$65k).
- Software architects ($95k, n=1,336) represent the highest-paid role with a large sample, making it a reliable reference point.

### Q6 — Does education level affect compensation?
- There is a visible gradient: Professional degree ($92.8k) > Master's ($81.2k) > Bachelor's ($80.3k) > Associate ($73.1k) > Some college ($71.4k) > Secondary ($60k).
- However, the boxes **heavily overlap** — the IQRs of Master's, Bachelor's, Associate, and Some college all span roughly the same range (~$40k–$120k). Only Professional stands slightly apart, and Secondary/Other sit noticeably lower.
- Master's and Bachelor's have the most high-end outliers (extending to $500k), likely reflecting higher representation in senior/high-paying roles rather than education itself.
- Primary has a very wide whisker range (few respondents, high variance), making it unreliable.

### Q6b — Does the education effect hold when controlling for experience?
- **Within each experience band, the education differences nearly vanish.** For example, at 16+ years of experience (green boxes), all education levels from Some college through Professional have similar medians in the $100k–$150k range.
- Experience is the dominant driver: the 0–5 y boxes (purple) are tightly packed around $30k–$60k regardless of education, while the 16+ y boxes (green) stretch to $100k–$200k for all levels.
- The small residual education premium (Professional slightly above others) is dwarfed by the experience effect.
- Implication: education may help with **entry-level placement** (first job, first salary band) but professional experience is what drives long-term earning trajectory.

### Q7 — Does remote work arrangement affect salary?
- Remote workers have the highest median ($90.8k), followed by Hybrid · flex ($81.2k), Your choice ($77.3k), Hybrid · office ($75k), and In-person ($49.6k).
- The violin shapes reveal that In-person has the **widest base at low salaries** (~$10k–$50k) and a much lower median, while Remote has a fatter body in the $50k–$150k range.
- However, all five violins have similar upper tails reaching $500k — high earners exist in every arrangement.
- The Remote vs In-person gap ($90.8k vs $49.6k) is almost certainly **confounded by geography**: remote workers are more likely to be in or employed by companies in high-income countries. This variable should be used cautiously in modelling without controlling for country.

### Q8 — Does company size correlate with higher compensation?
- A clear **monotonic increase**: freelancers ($50k) → <20 employees ($58.5k) → 20–99 ($70.8k) → 100–499 ($81.2k) → 500–999 ($81.9k) → 1k–5k ($87k) → 5k–10k ($88.6k) → 10k+ ($102k).
- The jump from freelancer to 10k+ employees is a **2× difference** in median salary.
- The most pronounced step-up happens between micro (<20) and mid-size (100–499) companies. Above 500 employees, the gains diminish.
- Larger companies also show more high-end outliers, with the 10k+ category having the densest scatter above $200k.
- The box widths (IQR) are similar across sizes, suggesting that within any company tier, variance is driven by other factors (geography, role, experience).

### Q9 — Are WorkExp and YearsCode redundant?
- Pearson r = 0.918, Spearman r = 0.906 — **near-perfect linear correlation**.
- The density heatmap shows a tight diagonal band from origin to ~(40, 45), confirming that for most developers, total coding years and professional experience track almost 1:1.
- The main deviation: `YearsCode` is systematically **higher** than `WorkExp` (the band sits above the y = x line), reflecting that many developers started coding before their first professional job (hobby coding, education).
- Modelling implication: using both features would introduce severe multicollinearity. Prefer `WorkExp` (more directly relevant) or engineer a `YearsCode − WorkExp` "pre-professional coding" feature.

### Q10 — Country × Education interaction on salary
- The Professional degree column is consistently the highest-paid education level within every country (US $175k, UK $95.3k, India $45.3k, France $73.6k).
- Geography dominates education: a US developer with only Secondary education ($130k) out-earns a French Professional degree holder ($73.6k) and virtually every education level in India.
- Within India the education gradient is steepest — Professional ($45.3k) earns **3.6× more** than Secondary ($10.6k) — suggesting education matters more in lower-income markets.
- In the US the gradient is nearly flat ($115k–$175k across all levels), reinforcing that experience and role matter more than education once geography is controlled.

### Q11 — What tech stacks are associated with the highest salaries?
- The **top 6 languages are identical** in both the highest and lowest salary quartiles: JavaScript, HTML/CSS, SQL, Python, TypeScript, Bash/Shell. Ubiquitous technologies dominate regardless of pay.
- The largest top-quartile skew is in **Bash/Shell** (noticeably more prevalent among high earners), consistent with infrastructure/DevOps roles that command higher pay.
- **HTML/CSS** and **JavaScript** are slightly more prevalent in the bottom quartile, consistent with lower-paid front-end or entry-level web roles.
- Niche languages like **Go**, **Rust**, and **Kotlin** appear almost exclusively in the top quartile but at low absolute share (~2–4 %), making them weak standalone predictors.
- Overall, language choice is a **noisy signal** — the same popular stack appears across the pay spectrum, and apparent differences are largely explained by the roles and geographies that favour those languages.

### Q12 — IC vs People Manager salary premium
- People managers have a moderately higher median ($87.6k vs $77.6k for ICs — a 13 % premium).
- However, the box plots overlap substantially: the IC IQR (~$50k–$100k) and PM IQR (~$60k–$125k) share most of their range.
- People managers show a denser cluster of outliers in the $350k–$500k band (upper whisker reaches ~$280k vs ~$240k for ICs), suggesting the management track has a higher ceiling.
- When stratified by experience (Q12b), the premium **grows with seniority**: at 0–5 years the two are nearly indistinguishable, but at 16+ years managers have a visibly higher box and more $400k+ outliers. This makes sense — early-career managers are rare and their premium hasn't compounded yet.

### Q13 — How does industry relate to compensation?
- **Fintech** ($95.9k) and **Healthcare** ($93.8k) lead, followed closely by Government ($88k) and Banking/Financial Services.
- **Software Development** ($71.9k) ranks last among the top 10, which is counterintuitive — but pure software companies employ more junior and front-end-heavy roles, pulling the median down.
- Software Development has by far the **longest outlier tail** (dense scatter reaching $500k), reflecting the extreme compensation at top tech companies while the median remains modest.
- Manufacturing ($81.2k) is compressed tightly around its median with few outliers — consistent with more standardised, less variable developer compensation.
- The gap between top and bottom industry is only ~1.3× ($95.9k vs $71.9k), making industry a **weaker signal than country, experience, or role**.

---

### Hypothesis assessment summary

| Feature | Signal | Key numbers |
|---------|--------|-------------|
| **Country** | Very strong | US $150k vs India $19.5k — 7.7× gap. Largest single factor. |
| **WorkExp** | Strong | Spearman r = 0.452. Median triples from 0–5 y ($40.6k) to 16+ y ($109k). |
| **DevType** | Strong | Eng. manager $135k vs academic researcher $43.3k — ~3× gap. |
| **EdLevel** | Moderate | Professional $92.8k vs Secondary $60k — 1.5× gap. Largely vanishes after controlling for experience. |
| **OrgSize** | Moderate | Freelancer $50k → 10k+ employees $102k — 2× monotonic increase. |
| **ICorPM** | Moderate | 13 % manager premium; grows with seniority. |
| **RemoteWork** | Weak–Moderate | Remote $90.8k vs In-person $49.6k — confounded by geography. |
| **Industry** | Moderate | Fintech $95.9k vs Software Dev $71.9k — only 1.3× spread; Software Dev has extreme upper tail. |
| **Tech stack** | Weak | Same top-6 languages in both salary quartiles. Niche languages too rare to be predictive. |

### Preliminary finding
The hypothesis is **supported**: professional experience, geographic location, and developer role are the dominant salary predictors. Education has a measurable but weaker effect that largely washes out after controlling for experience, and technology choices show the weakest independent signal.

### PPP insight
Country salary rankings shift meaningfully under PPP adjustment — Ukraine, India, Poland, and Romania all jump significantly. Important for the student advisory use-case: recommending countries by nominal salary alone is misleading.

### Data quality notes
- `WorkExp` and `YearsCode` are nearly redundant (Pearson r = 0.918) — use one or engineer a gap feature.
- Multi-select columns (`DevType`, `LanguageHaveWorkedWith`, etc.) need one-hot or frequency-based encoding for modelling.
- Cleaning removed rows with logical impossibilities (experience > age) — important for model integrity.

### Next steps (Sprint 2)
1. Feature engineering: encode multi-select columns, create interaction terms (Country × Experience), consider PPP-adjusted target.
2. Train/test split (80/20).
3. Baseline linear regression, then ensemble models.